# Differential spectra with explicit unavailable cases

SWAN notebook for the Upsilon early-science note. Keep the whole package together, check the shared profile, restart the kernel after updating the modules, then **Run All**.

Reads the selected reoptimization run's pT² and rapidity spectra, saved central toy sampling quantiles, full covariance and forward-folded Poisson slope fit. It does not rescale the saved inference or repeat the old bin-by-bin slope fit. The inset uses the saved asymmetric conditional profile endpoints; those are not automatically fully validated experimental uncertainties.

The pT² legend is at the upper left. Its tagged numerical spectrum and saved slope fit are read from the selected run; the inset layout, references and absent footer are retained. The rapidity panel retains the **flux-normalized companion** `(1/Phi_total) d sigma_ep/dy`, with the existing γp label as shorthand for this beam/selection-dependent convention. The primary ep values are also retained in the points CSV. This is not a universal fixed-W γp rapidity distribution.

Final case status is checked against `validation_status.csv` and retained in the outputs. `NOT_VALIDATED` results require explicit diagnostic permission in the shared profile; passing nominal extraction alone does not override that status. The displayed sampling quantiles are not confidence limits. Pure-state results omit backgrounds, mass-separation correlations and other uncertainties recorded in the source run.

Set `display_rebin.1S.y` to `"keep_outer_bins"` in the shared profile to merge the actual saved tagged bins into first/interior/last groups. Set it to `false` to retain the saved bins. With three or fewer source bins they remain unchanged. No old no-tag boundaries are assumed. This merges the original estimator, not a new three-bin response fit or optimization. Cross-section weights and the full covariance are propagated. Asymmetric sampling limits come from sums of joint Poisson toy fits replayed with the original seed and checked against the saved summaries. Marginal interval endpoints are never added. The first use can take a few minutes; results are cached in `OUTPUT_DIR/rebin_cache`. No ROOT files or full framework rerun are needed. The source run's actual validation status is preserved; display merging cannot promote it.

An explicitly completed `NO_FEASIBLE_SLOPE_BINNING` case appears in preflight with `available=False`, zero bins and `NOT_RUN`. It contributes no pT² points, slope or fit curve. Available rapidity results still render with their original final validation status. Missing files or rows for an otherwise available case remain errors. The availability CSV and audit retain every state/observable.

For the supplied tagged run, only 1S has a pT² spectrum and slope. All three rapidity spectra remain `NOT_VALIDATED` diagnostics. This update changes rendering only, not yields, binning or validation requirements.


**Repository profile:** copy `configs/plots_9x275.example.json` to `configs/plots.local.json` and edit that one file. Paths, runs, luminosity, selections, optional `display_rebin` and diagnostic permission are shared by all notebooks. See `docs/plotting.md`. Every execution creates a fresh output directory. The current figure layouts require all three states; the 1S-only 9×130 support run is reviewed through its status/audit outputs.


**Separate proton-tagged category.** The input reader requires the saved `jul26-reco-topology-v1` selection contract (topology 1/2 and finite positive tagged-proton momentum), applied to reconstruction only. Truth denominators and generated normalization remain unchanged. Run this notebook in a fresh kernel from this folder; do not mix modules or results with the original no-tag package.


In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from settings import load_profile, make_reoptimization_config, new_output_dir, apply_display_rebin, file_link_path
from upsilon_esr import preflight

# No ROOT or LaTeX installation is needed. If a Python dependency is missing,
# uncomment the next line, run it once, and restart the notebook kernel.
# %pip install -r requirements.txt


## Configuration

Notebooks 02/03 select one explicit reoptimization run for all states. Notebook 01 uses the separate tagged full-framework runs. This notebook requires a new tagged reoptimization run; it will stop if that run is absent.


In [ ]:
# All paths, run IDs, selection metadata and diagnostic permission come from one profile.
# Default: ../configs/plots.local.json. Alternatively set UPSILON_PLOT_PROFILE
# before starting the SWAN kernel, or give load_profile("/path/to/profile.json").
PROFILE = load_profile()
OUTPUT_DIR = new_output_dir(PROFILE, "differential")
cfg = make_reoptimization_config(profile=PROFILE, output_dir=OUTPUT_DIR)
apply_display_rebin(cfg, PROFILE, "differential")
print("Profile:", PROFILE["profile_path"])
print("Output directory:", OUTPUT_DIR)


## Validate inputs

Check the saved luminosity, beam/selection metadata, bin edges, conversions, full covariance and final case status. Missing or mismatched inputs raise an error; there is no fallback to old results. Nonvalidated but complete inference requires `allow_diagnostic_results: true` in the shared profile.

Explicit no-feasible cases are recorded as unavailable. Preflight still checks every remaining case; the first missing spectrum no longer hides later errors.


In [ ]:
display(preflight(cfg, section="differential"))

## Why a slope binning was rejected

These tables read the saved optimizer candidates, without repeating optimization. Failure counts overlap because one candidate can fail several requirements. A candidate failing **only** `expected_signal_reco_count` passes the other binning gates, so its count requirement can improve with luminosity. That still does not guarantee subsequent inference/coverage validation. The actual saved thresholds are authoritative. No candidate is accepted or relabeled by this diagnostic.


In [ ]:
from collections import Counter
import json

base = Path(cfg.reoptimization_run) / "artifacts/reoptimization"
optimization_diagnostics = {}
rejection_rows = []
for state in ("1S", "2S", "3S"):
    entry = json.loads((base / state / "pt2/result.json").read_text())
    opt = entry["optimization"]
    candidates = opt.get("candidates", [])
    counts = Counter(reason for c in candidates for reason in c["failure_reasons"])
    count_only = [c for c in candidates
                  if set(c["failure_reasons"]) == {"expected_signal_reco_count"}]
    best = max(count_only, key=lambda c: c["min_expected_reco"]) if count_only else None
    summary = {
        "overall_status":entry["overall_status"],
        "thresholds":opt.get("thresholds"),
        "candidate_count":len(candidates),
        "failed_candidates_per_requirement":dict(counts),
        "yield_only_failed_candidates":len(count_only),
        "best_yield_only_candidate":None if best is None else {
            k:best.get(k) for k in ("edges", "min_expected_reco", "min_purity",
                                   "min_selected_stability", "condition_number")},
    }
    optimization_diagnostics[state] = summary
    print(state, json.dumps(summary, indent=2))
    for reason, number in counts.items():
        rejection_rows.append({"state":state, "failed_requirement":reason,
                               "rejected_candidates":number})
rejection_table = pd.DataFrame(rejection_rows,
    columns=["state", "failed_requirement", "rejected_candidates"])
display(rejection_table)
(OUTPUT_DIR / "optimization_rejection_summary.json").write_text(
    json.dumps(optimization_diagnostics, indent=2))
rejection_table.to_csv(OUTPUT_DIR / "optimization_rejections.csv", index=False)


## Publication typography

The shared style is tuned for the single-column figures in the note. Source sizes are 16 pt for axis titles, 14.5 pt for ticks, 14 pt for legends, 13 pt for annotations, and 12.5–13.5 pt inside the slope inset. These become approximately 5.7–7.4 pt at the paper column width. The sizes are centralized in `upsilon_esr/common.py` as `FONT_SIZES`. Replacing the plotting modules requires a kernel restart before rerunning.


## Render the figure

In [ ]:
from upsilon_esr.differential import plot_differential

FIT_RANGE_GEV2 = (0.0, 1.0) # must match the saved forward-fit range; no new fit is performed
fig, audit = plot_differential(cfg, fit_range=FIT_RANGE_GEV2)
plt.show()


## Numerical check

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "upsilon_es_differential_availability.csv"))
display(pd.read_csv(OUTPUT_DIR / "upsilon_es_slopes.csv"))
for state, result in audit["slope_fits"].items():
    print(state, "used fit bins:", result.get("used_fit_bins_GeV2", result))
for message in audit["warnings"]:
    print("Note:", message)


## Saved outputs

PNG/PDF, points, slope and availability tables, and the complete audit go to the fresh directory printed below. Input framework results and earlier plots are unchanged.


In [ ]:
print("Output directory:", OUTPUT_DIR.resolve())
for name in ("upsilon_es_differential.pdf", "upsilon_es_differential.png",
             "upsilon_es_differential_availability.csv", "upsilon_es_slopes.csv",
             "upsilon_es_differential_audit.json", "optimization_rejection_summary.json"):
    display(FileLink(file_link_path(OUTPUT_DIR / name)))
